In [ ]:
%pip install numpy scikit-learn matplotlib google-cloud-storage google-auth

In [ ]:
import os
import sys

REPO_URL   = "https://github.com/payamdav/pycrypto.git"
REPO_NAME  = "pycrypto"
BRANCH     = "claude/quantile-transform-effect-notebook-56hzy5"

if not os.path.exists(REPO_NAME):
    !git clone -b {BRANCH} {REPO_URL}

REPO_PATH = os.path.abspath(REPO_NAME)
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import QuantileTransformer

sys.path.insert(0, "pycrypto/packages/tools/google_cloud_storage_tools")
from gcs_tools import gcs_json_key_file, read_file

BUCKET = "payamdprojectbucket"

gcs_json_key_file()
print("GCS ready")

In [ ]:
def plot_asset(asset):
    print(f"Loading {asset.upper()} ...")
    raw = read_file(BUCKET, f"fl_data_{asset}")
    fl  = np.load(io.BytesIO(raw))
    n   = fl.shape[0]
    print(f"  shape={fl.shape}  |  {n:,} observations")

    # l_e_vwap_0..3 live at columns 52-55
    labels = fl[:, 52:56].copy()   # (N, 4)

    # Fit one QuantileTransformer per column (uniform output in [0, 1])
    qts       = []
    labels_qt = np.empty_like(labels)
    for i in range(4):
        qt = QuantileTransformer(
            n_quantiles=min(1000, n),
            output_distribution="uniform",
            random_state=42,
        )
        labels_qt[:, i] = qt.fit_transform(labels[:, i : i + 1]).ravel()
        qts.append(qt)

    probe      = np.round(np.arange(-1.0, 1.01, 0.2), 1)   # 11 values
    lnames     = [
        "l_e_vwap_0 (1 h)",
        "l_e_vwap_1 (2 h)",
        "l_e_vwap_2 (3 h)",
        "l_e_vwap_3 (4 h)",
    ]

    # 3 rows × 4 cols:  row 0 = original histo, row 1 = QT histo, row 2 = table
    fig = plt.figure(figsize=(24, 22))
    fig.suptitle(
        f"Quantile Transform Effect — {asset.upper()}",
        fontsize=17, fontweight="bold", y=0.995,
    )
    gs = fig.add_gridspec(3, 4, height_ratios=[3, 3, 4.5], hspace=0.50, wspace=0.35)

    for i, lname in enumerate(lnames):
        col_orig = labels[:, i]
        col_qt   = labels_qt[:, i]

        # ── original histogram ───────────────────────────────────────────────
        ax0 = fig.add_subplot(gs[0, i])
        ax0.hist(col_orig, bins=200, color="#2471A3", edgecolor="none", alpha=0.9)
        ax0.set_title(f"Original\n{lname}", fontsize=10, fontweight="bold")
        ax0.set_xlabel("Value [-1, 1]", fontsize=9)
        ax0.set_ylabel("Count", fontsize=9)
        ax0.grid(True, alpha=0.3, linewidth=0.5)
        ax0.tick_params(labelsize=8)

        # ── quantile-transformed histogram ───────────────────────────────────
        ax1 = fig.add_subplot(gs[1, i])
        ax1.hist(col_qt, bins=100, color="#E67E22", edgecolor="none", alpha=0.9)
        ax1.set_title(f"QuantileTransform\n{lname}", fontsize=10, fontweight="bold")
        ax1.set_xlabel("Quantile [0, 1]", fontsize=9)
        ax1.set_ylabel("Count", fontsize=9)
        ax1.grid(True, alpha=0.3, linewidth=0.5)
        ax1.tick_params(labelsize=8)

        # ── mapping table ────────────────────────────────────────────────────
        ax2 = fig.add_subplot(gs[2, i])
        ax2.axis("off")
        ax2.set_title(f"Input → Transformed\n{lname}", fontsize=10, fontweight="bold")

        probe_qt   = qts[i].transform(probe.reshape(-1, 1)).ravel()
        table_data = [[f"{p:.1f}", f"{t:.4f}"] for p, t in zip(probe, probe_qt)]

        tbl = ax2.table(
            cellText=table_data,
            colLabels=["Input", "Transformed"],
            cellLoc="center",
            loc="center",
            bbox=[0.10, 0.0, 0.80, 1.0],
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9.5)

        # header style
        for col in range(2):
            cell = tbl[0, col]
            cell.set_facecolor("#1A5276")
            cell.set_text_props(color="white", fontweight="bold")

        # alternating row colours
        for row in range(1, len(probe) + 1):
            bg = "#D6EAF8" if row % 2 == 0 else "#FDFEFE"
            for col in range(2):
                tbl[row, col].set_facecolor(bg)

    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.show()
    print(f"{asset.upper()} done.\n")


## BTCUSDT

In [ ]:
plot_asset("btcusdt")

## ETHUSDT

In [ ]:
plot_asset("ethusdt")

## ADAUSDT

In [ ]:
plot_asset("adausdt")

## XRPUSDT

In [ ]:
plot_asset("xrpusdt")

## TRUMPUSDT

In [ ]:
plot_asset("trumpusdt")

## VINEUSDT

In [ ]:
plot_asset("vineusdt")

## DOGEUSDT

In [ ]:
plot_asset("dogeusdt")